# Experiment: App Data Trial Cleaning Parameter Exploration

Objective:
- 评估 `"相对时间(秒)"` 作为试次清洗参数时，不同阈值对应的无效试次规模。
- 输出三类汇总：`threshold=0.1`、`threshold=0.2` 的低阈值无效试次统计，以及每个 sheet 内 `mean + 3 s.d.` 高阈值试次统计。
- 数据范围为 `data/raw/*/app_data/*.xlsx` 下全部原始 app 行为文件。


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

from pathlib import Path
import statistics
import warnings

import openpyxl
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "ARCHITECTURE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from the current working directory.")


RELATIVE_TIME_COL = "相对时间(秒)"
CORRECT_ANSWER_COL = "正式阶段正确答案"
RESPONSE_COL = "正式阶段被试按键"
LOW_THRESHOLDS = (0.1, 0.2)
REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = REPO_ROOT / "data" / "raw"
APP_DATA_FILES = sorted(DATA_ROOT.glob("*/app_data/*.xlsx"))

len(APP_DATA_FILES)


## Plan

- 将每个 Excel 文件视为一个被试，将每个 sheet 视为一个任务表。
- 对 `threshold=0.1` 与 `threshold=0.2`，分别统计：
  - 平均每个被试有多少个 sheet 出现了无效试次；
  - 平均每个 sheet 有多少个无效试次。
  - 无效试次的平均正确率各自是多少。
- 再对每个 sheet 计算 `mean + 3 s.d.`，统计超过该上界的试次数，并汇总平均每个 sheet 有多少个这类高阈值无效试次。
- 读取时同时使用 `相对时间(秒)`、`正式阶段正确答案`、`正式阶段被试按键` 三列；缺失值与非数值将按当前统计口径忽略或视为错误。


In [ ]:
# Define parameters and lightweight helpers
def threshold_label(threshold: float) -> str:
    return str(threshold).replace(".", "p")


def extract_site(workbook_path: Path) -> str:
    return workbook_path.parent.parent.name


def normalize_response(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def iter_sheet_trial_records(workbook_path: Path):
    workbook = openpyxl.load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        for sheet_name in workbook.sheetnames:
            worksheet = workbook[sheet_name]
            rows = worksheet.iter_rows(values_only=True)
            try:
                header = next(rows)
            except StopIteration:
                continue

            required_columns = (RELATIVE_TIME_COL, CORRECT_ANSWER_COL, RESPONSE_COL)
            if header is None or any(column not in header for column in required_columns):
                continue

            relative_time_index = header.index(RELATIVE_TIME_COL)
            correct_answer_index = header.index(CORRECT_ANSWER_COL)
            response_index = header.index(RESPONSE_COL)
            trial_records = []
            for row in rows:
                if row is None:
                    trial_records.append({"relative_time": None, "is_correct": 0})
                    continue
                value = row[relative_time_index] if relative_time_index < len(row) else None
                try:
                    relative_time = float(value)
                except (TypeError, ValueError):
                    relative_time = None
                correct_answer = normalize_response(
                    row[correct_answer_index] if correct_answer_index < len(row) else None
                )
                response = normalize_response(
                    row[response_index] if response_index < len(row) else None
                )
                trial_records.append(
                    {
                        "relative_time": relative_time,
                        "is_correct": int(
                            correct_answer is not None
                            and response is not None
                            and correct_answer == response
                        ),
                    }
                )

            yield sheet_name, trial_records
    finally:
        workbook.close()


def analyze_workbook(workbook_path: Path) -> list[dict[str, object]]:
    subject_id = workbook_path.stem.replace("_GameData", "")
    site = extract_site(workbook_path)
    records = []

    for sheet_name, trial_records in iter_sheet_trial_records(workbook_path):
        relative_times = [trial["relative_time"] for trial in trial_records]
        valid_values = [value for value in relative_times if value is not None]
        if not valid_values:
            continue

        mean_value = statistics.mean(valid_values)
        sd_value = statistics.stdev(valid_values) if len(valid_values) > 1 else 0.0
        upper_cutoff = mean_value + 3 * sd_value

        record = {
            "site": site,
            "subject_id": subject_id,
            "sheet": sheet_name,
            "n_trials": len(trial_records),
            "n_nonmissing_relative_time": len(valid_values),
            "mean_relative_time": mean_value,
            "sd_relative_time": sd_value,
            "upper_cutoff_mean_plus_3sd": upper_cutoff,
            "invalid_gt_mean_plus_3sd": sum(
                value is not None and value > upper_cutoff for value in relative_times
            ),
        }

        for threshold in LOW_THRESHOLDS:
            label = threshold_label(threshold)
            invalid_trials = [
                trial
                for trial in trial_records
                if trial["relative_time"] is not None and trial["relative_time"] < threshold
            ]
            invalid_count = len(invalid_trials)
            record[f"invalid_lt_{label}"] = invalid_count
            record[f"has_invalid_lt_{label}"] = int(invalid_count > 0)
            record[f"correct_invalid_lt_{label}"] = sum(
                trial["is_correct"] for trial in invalid_trials
            )

        records.append(record)

    return records


In [ ]:
# Run the workbook-level analysis
sheet_records: list[dict[str, object]] = []
for workbook_path in APP_DATA_FILES:
    sheet_records.extend(analyze_workbook(workbook_path))

sheet_level = pd.DataFrame(sheet_records)
sheet_level.head()


In [ ]:
# Summaries requested by the analysis question
threshold_summary_rows = []
for threshold in LOW_THRESHOLDS:
    label = threshold_label(threshold)
    invalid_count_col = f"invalid_lt_{label}"
    invalid_flag_col = f"has_invalid_lt_{label}"

    threshold_summary_rows.append(
        {
            "threshold": threshold,
            "avg_invalid_sheets_per_subject": sheet_level.groupby("subject_id")[invalid_flag_col].sum().mean(),
            "avg_invalid_trials_per_sheet": sheet_level[invalid_count_col].mean(),
            "avg_accuracy_among_invalid_trials": (
                sheet_level[f"correct_invalid_lt_{label}"].sum() / sheet_level[invalid_count_col].sum()
                if sheet_level[invalid_count_col].sum() > 0
                else float("nan")
            ),
        }
    )

threshold_summary = pd.DataFrame(threshold_summary_rows)
outlier_summary = pd.DataFrame(
    [
        {
            "rule": "relative_time > mean + 3 s.d. (within sheet)",
            "avg_invalid_trials_per_sheet": sheet_level["invalid_gt_mean_plus_3sd"].mean(),
        }
    ]
)

display(threshold_summary)
display(outlier_summary)


In [ ]:
# Optional exploration: which sheets are most affected?
sheet_breakdown = (
    sheet_level.groupby("sheet")[["invalid_lt_0p1", "invalid_lt_0p2", "invalid_gt_mean_plus_3sd"]]
    .mean()
    .sort_values("invalid_lt_0p2", ascending=False)
)
sheet_breakdown.head(10)


## Results

- 当前全量数据（738 个 workbook，12,830 个 sheet）下：
  - `threshold=0.1` 时，平均每个被试有 `1.403794` 个 sheet 出现无效试次；平均每个 sheet 有 `0.235464` 个无效试次；无效试次平均正确率为 `0.501821`。
  - `threshold=0.2` 时，平均每个被试有 `3.855014` 个 sheet 出现无效试次；平均每个 sheet 有 `1.236555` 个无效试次；无效试次平均正确率为 `0.521904`。
  - 若将每个 sheet 内 `相对时间(秒) > mean + 3 s.d.` 的试次视为高阈值无效试次，则平均每个 sheet 有 `1.001403` 个无效试次。
- `threshold_summary` 汇总了两个低阈值方案下的核心统计量，包括无效试次平均正确率。
- `outlier_summary` 汇总了每个 sheet 内按 `mean + 3 s.d.` 标记的高阈值无效试次数。
- `sheet_breakdown` 可用于查看哪些任务表对阈值更敏感。


## Next steps

- 如果要把该规则纳入正式预处理脚本，可将当前 notebook 中的 sheet 级汇总逻辑抽取为 `src/behavior/app/clean.py` 的试次级清洗函数。
- 如果要比较不同任务或站点的清洗敏感性，可在 `sheet_level` 基础上继续按 `site` 或 `sheet` 聚合。
